# Red Column Issues:
A public/private test case might be all failing and the evolution stalls here (disengagement)

This notebook is to see whether it improves given the feedback

In [33]:
from common.coevolution import lcb_dataset
problems = lcb_dataset.load_code_generation_dataset(
    release_version="release_v6",
    start_date="2025-03-01",
    end_date="2025-05-10",
    difficulty=lcb_dataset.Difficulty.HARD,
)

problem_id = "arc194_c"
problem = next((p for p in problems if p.question_id == problem_id), None)
if problem is None:
    raise ValueError(f"Problem with ID {problem_id} not found.")    

13:34:52 | INFO     | [SETUP   ] | common.coevolution.lcb_dataset:load_code_generation_dataset:127 - Using 'test' split of the dataset.
13:35:18 | INFO     | [SETUP   ] | common.coevolution.lcb_dataset:load_code_generation_dataset:141 - Filtered problems by difficulty: hard
13:35:18 | INFO     | [SETUP   ] | common.coevolution.lcb_dataset:load_code_generation_dataset:151 - Loaded 37 problems


# Initialize Code and Test Populations
use the public test case for the test population

In [34]:
from common.coevolution.core.individual import TestIndividual
from common.coevolution.core.population import TestPopulation
from common.coevolution.core.interfaces import Operations
from common.coevolution import lcb_dataset
from common.code_preprocessing.transformation import extract_test_methods_code
from common.coevolution.core.mock import MockPareto, MockTestBlockRebuilder
# Build the test class block from dataset test cases
test_class_block = lcb_dataset.LCBDatasetTestBlockBuilder.build_test_class_block(
    problem.public_test_cases, problem.starter_code
)

# Extract individual test methods
test_methods = extract_test_methods_code(test_class_block)

# Create test individuals with fixed probability
FIXED_TEST_PROBABILITY = 1.0
test_individuals = [
    TestIndividual(
        snippet=method,
        probability=FIXED_TEST_PROBABILITY,
        creation_op=Operations.INITIAL,
        generation_born=0,
        parent_ids=[],
    )
    for method in test_methods
]

# Create and return the test population
test_population = TestPopulation(
    individuals=test_individuals,
    pareto=MockPareto(),
    test_block_rebuilder=MockTestBlockRebuilder(),
    test_class_block=test_class_block,
    generation=0,
)

13:35:18 | DEBUG    | [SETUP   ] | common.coevolution.core.individual:__init__:90 - Created new <TestIndividual id=T6 gen=0 op=initial prob=1.00>
13:35:18 | DEBUG    | [SETUP   ] | common.coevolution.core.individual:__init__:90 - Created new <TestIndividual id=T7 gen=0 op=initial prob=1.00>
13:35:18 | DEBUG    | [SETUP   ] | common.coevolution.core.individual:__init__:90 - Created new <TestIndividual id=T8 gen=0 op=initial prob=1.00>
13:35:18 | DEBUG    | [SETUP   ] | common.coevolution.core.interfaces:__init__:479 - Initialized TestPopulation with 3 individuals, gen 0


In [12]:
from common.coevolution.core.individual import CodeIndividual
from common.coevolution.core.population import CodePopulation

C1 = """
class Solution:
    def sol(self, input_str: str) -> str:
        data = list(map(int, input_str.split()))
        it = iter(data)
        N = next(it)
        A = [next(it) for _ in range(N)]
        B = [next(it) for _ in range(N)]
        C = [next(it) for _ in range(N)]
        # Strategy: flip some 1->0 that reduce future costs before doing 0->1 that raise them.
        ones_sum = sum(C[i] for i in range(N) if A[i] == 1)
        idxs_to_zero = [i for i in range(N) if A[i] == 1 and B[i] == 0]
        idxs_to_one = [i for i in range(N) if A[i] == 0 and B[i] == 1]
        # Flip those 1->0 in descending C to maximize immediate reduction early.
        idxs_to_zero.sort(key=lambda i: -C[i])
        total = 0
        for i in idxs_to_zero:
            A[i] = 0
            ones_sum -= C[i]
            total += ones_sum
        # Then flip 0->1 in ascending C to minimize increments when many ones exist
        idxs_to_one.sort(key=lambda i: C[i])
        for i in idxs_to_one:
            A[i] = 1
            ones_sum += C[i]
            total += ones_sum
        return str(total)
"""


code_individual = CodeIndividual(
    snippet=C1,
    creation_op=Operations.INITIAL,
    generation_born=0,
    parent_ids=[],
    probability=0.75
)

code_population = CodePopulation([code_individual], 0)
print(f"Code Population:\n{code_population}")

23:56:18 | DEBUG    | [SETUP   ] | common.coevolution.core.individual:__init__:42 - Created new <CodeIndividual id=C1 gen=0 op=initial prob=0.75>
23:56:18 | DEBUG    | [SETUP   ] | common.coevolution.core.interfaces:__init__:479 - Initialized CodePopulation with 1 individuals, gen 0


Code Population:
<CodePopulation gen=0 size=1 avg_prob=0.7500>


# Execute

In [13]:
from common.coevolution.execution import ExecutionSystem
from common.sandbox import create_safe_test_environment

sandbox = create_safe_test_environment()
exec_system = ExecutionSystem(enable_multiprocessing=True, num_workers=10)
exec_results = exec_system.execute_tests(code_population, test_population, sandbox=sandbox)
observation_matrix = exec_system.build_observation_matrix(code_population, test_population, exec_results)
print(f"Observation Matrix:\n{observation_matrix}")

23:56:18 | INFO     | [SETUP   ] | common.coevolution.execution:execute_tests:153 - Executing code against tests: 1 code × 3 tests = 3 evaluations using 1 workers.
23:56:18 | DEBUG    | [SETUP   ] | common.coevolution.execution:_execute_sequentially:223 - Running sequential execution (no multiprocessing)
23:56:18 | INFO     | [SETUP   ] | common.coevolution.logging_utils:setup_logging:120 - Logging configured. Console level: DEBUG, File level: TRACE.
23:56:18 | INFO     | [SETUP   ] | common.coevolution.logging_utils:setup_logging:123 - Detailed logs will be written to logs/coevolution_run_{time:YYYYMMDD}.log
23:56:18 | DEBUG    | [SETUP   ] | common.sandbox:execute_test_script:691 - SafeCodeSandbox: executing test script (len=1709)
23:56:19 | DEBUG    | [SETUP   ] | common.sandbox:analyze_pytest_xml:150 - Analyzing pytest XML output: xml_available=True, success=False, return_code=1
23:56:19 | DEBUG    | [SETUP   ] | common.sandbox:execute_test_script:765 - SafeCodeSandbox: test script

Observation Matrix:
[[1 1 0]]


# Build feedback

In [14]:
from common.coevolution.feedback import CodeFeedbackGenerator

code_feedback_generator = CodeFeedbackGenerator()
feedback = code_feedback_generator.generate_feedback(observation_matrix, exec_results, test_population, 0)

print(f"Feedback:\n{feedback}")

23:56:19 | DEBUG    | [SETUP   ] | common.coevolution.feedback:generate_feedback:160 - Code 0: Generating feedback - 1 failures, 2 passed out of 3 total
23:56:19 | DEBUG    | [SETUP   ] | common.coevolution.feedback:generate_feedback:198 - Code 0: Processed 1 failed and 2 passed test entries
23:56:19 | DEBUG    | [SETUP   ] | common.coevolution.feedback:generate_feedback:221 - Code 0: Generated feedback (1353 chars)


Feedback:
This code passed 2 out of 3 tests:

The following test(s) failed:

1. test_case_3
Test code:
```python
def test_case_3(self):
    input_str = '20\n1 1 1 1 0 0 1 1 0 0 0 1 0 1 0 1 1 0 1 0\n0 0 0 1 1 1 0 1 1 0 0 0 0 0 0 1 0 1 0 0\n52 73 97 72 54 15 79 67 13 55 65 22 36 90 84 46 1 2 27 8'
    expected_output = '2867'
    self.assertEqual(self.solution.sol(input_str), expected_output)
```
Error: self = <tmpmyxk1_ce.TestSolution testMethod=test_case_3>

    def test_case_3(self):
        input_str = '20\n1 1 1 1 0 0 1 1 0 0 0 1 0 1 0 1 1 0 1 0\n0 0 0 1 1 1 0 1 1 0 0 0 0 0 0 1 0 1 0 0\n52 73 97 72 54 15 79 67 13 55 65 22 36 90 84 46 1 2 27 8'
        expected_output = '2867'
>       self.assertEqual(self.solution.sol(input_str), expected_output)
E       AssertionError: '3300' != '2867'
E       - 3300
E       + 2867

/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpmyxk1_ce.py:48: AssertionError


The following test(s) passed:

1. test_case_1
Test code:
```python
def test_case_1(

In [ ]:
from common.coevolution.prompt_templates import EDIT_CODE
from common.llm_client import create_llm_client
from common.code_preprocessing.extraction import extract_code_block_from_response
llm_client = create_llm_client(provider="openai", model="gpt-5-mini")

prompt = EDIT_CODE.format(
    question_content=problem.question_content,
    individual=code_individual.snippet,
    feedback=feedback,
    starter_code=problem.starter_code, 
)


response = llm_client.generate(prompt)
print(f"LLM Response:\n{response}")

code_block = extract_code_block_from_response(response)

00:03:51 | INFO     | [SETUP   ] | common.llm_client:create_llm_client:371 - Creating LLM client: provider=openai, model=gpt-5-mini
00:03:51 | DEBUG    | [SETUP   ] | common.llm_client:__init__:43 - Initialized LLM client: model=gpt-5-mini, max_tokens=1000000, limit_enabled=True
00:03:52 | DEBUG    | [SETUP   ] | common.llm_client:__init__:247 - Initialized OpenAIClient with model: gpt-5-mini, reasoning_effort: minimal
00:03:52 | INFO     | [SETUP   ] | common.llm_client:create_llm_client:403 - Successfully created OpenAIClient
00:03:52 | DEBUG    | [SETUP   ] | common.llm_client:generate:256 - OpenAIClient generating with model: gpt-5-mini, reasoning_effort: minimal
00:04:14 | DEBUG    | [SETUP   ] | common.llm_client:generate:276 - Generated 3831 characters


Edited Code Block:
class Solution:
    def sol(self, input_str: str) -> str:
        # Parse input
        data = list(map(int, input_str.split()))
        it = iter(data)
        N = next(it)
        A = [next(it) for _ in range(N)]
        B = [next(it) for _ in range(N)]
        C = [next(it) for _ in range(N)]

        # We need to transform A to B by flipping bits one at a time.
        # Each flip costs sum(A_k * C_k) after the flip.
        #
        # Observations / standard reduction:
        # - Only indices where A_i != B_i need their bit toggled an odd number of times.
        # - Indices where A_i == B_i must be toggled an even number of times (so effectively zero
        #   times in an optimal solution because every extra pair of flips would add cost).
        # So we need to decide an order to perform the required flips (each required index flipped exactly once).
        #
        # Let S be the set of indices we need to flip (A_i != B_i).
        # When flipping index 

In [32]:
edited_code_individual = CodeIndividual(
    snippet=code_block,
    creation_op=Operations.EDIT,
    generation_born=1,
    parent_ids=[code_individual.id],
    probability=0.75
)

code_population.set_next_generation([edited_code_individual])
new_exec_results = exec_system.execute_tests(code_population, test_population, sandbox=sandbox)
new_observation_matrix = exec_system.build_observation_matrix(code_population, test_population, new_exec_results)

print(f"New Observation Matrix:\n{new_observation_matrix}")

for test_result in new_exec_results[0].test_results:
    print(f"{test_result.status}")
    print(f"{test_result.details}")


00:13:04 | DEBUG    | [SETUP   ] | common.coevolution.core.individual:__init__:42 - Created new <CodeIndividual id=C14 gen=1 op=edit prob=0.75>
00:13:04 | DEBUG    | [SETUP   ] | common.coevolution.core.interfaces:set_next_generation:563 - Gen 10 -> 11: Kept 0 elites, Added 1 new offspring, Removed 1 individuals.
00:13:04 | INFO     | [SETUP   ] | common.coevolution.core.interfaces:set_next_generation:585 - Advanced CodePopulation to gen 11: size 1 -> 1, avg_prob 0.7500 -> 0.7500
00:13:04 | INFO     | [SETUP   ] | common.coevolution.execution:execute_tests:153 - Executing code against tests: 1 code × 3 tests = 3 evaluations using 1 workers.
00:13:04 | DEBUG    | [SETUP   ] | common.coevolution.execution:_execute_sequentially:223 - Running sequential execution (no multiprocessing)
00:13:04 | INFO     | [SETUP   ] | common.coevolution.logging_utils:setup_logging:120 - Logging configured. Console level: DEBUG, File level: TRACE.
00:13:04 | INFO     | [SETUP   ] | common.coevolution.loggin

New Observation Matrix:
[[1 1 0]]
passed
None
passed
None
failed
self = <tmpk8768eom.TestSolution testMethod=test_case_3>

    def test_case_3(self):
        input_str = '20\n1 1 1 1 0 0 1 1 0 0 0 1 0 1 0 1 1 0 1 0\n0 0 0 1 1 1 0 1 1 0 0 0 0 0 0 1 0 1 0 0\n52 73 97 72 54 15 79 67 13 55 65 22 36 90 84 46 1 2 27 8'
        expected_output = '2867'
>       self.assertEqual(self.solution.sol(input_str), expected_output)
E       AssertionError: '3809' != '2867'
E       - 3809
E       + 2867

/var/folders/bl/ydbmym3d04qb5y3mvth453g40000gp/T/tmpk8768eom.py:47: AssertionError
